# 예제 04. 금융 시계열 데이터 분석과 추세 시각화
### 시간의 흐름에 따른 금융지표 변화 이해 · 1일 차 · 1.5시간

> **실행 방법**: 위에서부터 순서대로 실행하세요 (`Shift + Enter`).

| 항목 | 내용 |
|---|---|
| 데이터 | 한국은행 경제통계시스템(ECOS) 예금은행 수신금리 월별 시계열 · 2016년 1월 ~ 2026년 6월 (126개월) |
| 시계열 코드 | 통계표 121Y002 · 항목 BEABAA2 |
| 오늘의 결과물 | 월별 금리 추세 · 이동평균 그래프 |

**오늘 해결할 질문**: "지난 10년 동안 예금금리는 언제 오르고 언제 내렸을까?"

> 이 금리는 **예금은행(시중은행 등)의 평균 금리**입니다. 저축은행이 실제로 주는 금리와는 다르며, 시장 금리의 큰 흐름을 보는 자료로 씁니다.

## 0. 실습 준비

아래 셀을 실행하면 실습 데이터를 **GitHub에서 자동으로 내려받습니다.** (강의장 인터넷이 막혀 **[파일 선택]** 버튼이 나오면, 강사가 배포한 파일을 고르면 됩니다.)
`데이터 준비 완료: True`가 나오면 성공입니다.

In [ ]:
import os
import urllib.request
import pandas as pd
import plotly.express as px

DATA_URL = "https://raw.githubusercontent.com/jetsonai/AIFinancialDataAnalysis/main/Day1/data/"
FILE = "deposit_rate_monthly.csv"
if not os.path.exists(FILE):          # 파일이 없으면 GitHub에서 내려받습니다
    try:
        urllib.request.urlretrieve(DATA_URL + FILE, FILE)
    except Exception:                 # 인터넷이 막혔을 때만 직접 올리기
        from google.colab import files
        files.upload()
print("데이터 준비 완료:", os.path.exists(FILE))

## 04-1. 월별 금리 데이터 읽기

`parse_dates=["date"]`를 넣으면 날짜 글자를 **날짜**로 읽어 줍니다. 그래야 시간 순서대로 계산하고 그릴 수 있습니다.
컬럼 이름은 알아보기 쉽게 한국어로 바꿉니다.

In [ ]:
df = pd.read_csv(FILE, parse_dates=["date"])
df = df.rename(columns={"date": "월", "deposit_rate_pct": "금리", "mom_change_pp": "전월대비"})
df.head()

| 컬럼 | 뜻 | 단위 |
|---|---|---|
| 월 | 기준 월 (매월 1일로 표시) | - |
| 금리 | 예금은행 수신금리 | 연 % |
| 전월대비 | 지난달보다 오르내린 폭 | %p (퍼센트포인트) |
| series_code | 한국은행 ECOS 시계열 코드 | - |

> 💡 금리가 2.0%에서 2.5%로 오르면 **0.5%p** 오른 것입니다. "25% 올랐다"와는 다른 말이니 구분해서 씁니다.

## 04-2. 기간과 개수 확인

In [ ]:
print("시작:", df["월"].min().date())
print("끝  :", df["월"].max().date())
print("개월 수:", len(df))
df.tail()

## 04-3. 금리 추세 그래프

`px.line()`은 시간에 따른 변화를 선으로 그립니다. 그래프 아래쪽을 드래그하면 기간을 확대할 수 있고, 더블클릭하면 원래대로 돌아옵니다.

In [ ]:
fig = px.line(df, x="월", y="금리", title="예금은행 수신금리 추이 (연 %)")
fig.show()

## 04-4. 가장 높았던 달, 가장 낮았던 달

`idxmax()`는 가장 큰 값이 있는 **행 번호**를, `loc[행 번호]`는 그 행 전체를 꺼내 줍니다.

In [ ]:
최고 = df.loc[df["금리"].idxmax()]
최저 = df.loc[df["금리"].idxmin()]
print("최고:", 최고["월"].strftime("%Y년 %m월"), 최고["금리"], "%")
print("최저:", 최저["월"].strftime("%Y년 %m월"), 최저["금리"], "%")

In [ ]:
fig = px.line(df, x="월", y="금리", title="예금은행 수신금리 — 최고점과 최저점")
fig.add_annotation(x=최고["월"], y=최고["금리"], text=f"최고 {최고['금리']}%")
fig.add_annotation(x=최저["월"], y=최저["금리"], text=f"최저 {최저['금리']}%")
fig.show()

## 04-5. 전월 대비 변화량 직접 계산하기

`diff()`는 **바로 앞 행과의 차이**를 계산합니다. 첫 달은 앞 달이 없어서 빈 값(NaN)이 됩니다.
원본에 있는 `전월대비` 값과 같은지 확인해 봅니다.

In [ ]:
df["변화량"] = df["금리"].diff().round(2)
df[["월", "금리", "변화량", "전월대비"]].head()

In [ ]:
print("직접 계산과 원본의 최대 차이:", (df["변화량"] - df["전월대비"]).abs().max())

## 04-6. 상승 · 하락 구간 확인

변화량이 0보다 크면 상승, 작으면 하락으로 표시하고 색을 나눠 막대로 그립니다. (빨강 = 상승, 파랑 = 하락)
첫 달(2016년 1월)은 비교할 앞 달이 없어서 보합으로 남습니다. 그래서 보합 8개 중 1개는 첫 달입니다.

In [ ]:
df["방향"] = "보합"
df.loc[df["변화량"] > 0, "방향"] = "상승"
df.loc[df["변화량"] < 0, "방향"] = "하락"

df["방향"].value_counts()

In [ ]:
fig = px.bar(df, x="월", y="변화량", color="방향",
             color_discrete_map={"상승": "#E5484D", "하락": "#3776AB", "보합": "#94A3B8"},
             title="전월 대비 금리 변화량 (%p)")
fig.show()

## 04-7. 가장 크게 오른 달 · 내린 달

`nlargest(5, "항목")`은 값이 큰 순서로 5개, `nsmallest`는 작은 순서로 5개를 고릅니다.

In [ ]:
df.nlargest(5, "변화량")[["월", "금리", "변화량"]]

In [ ]:
df.nsmallest(5, "변화량")[["월", "금리", "변화량"]]

> 💡 **핵심 정리** — 가장 크게 오른 5개 달 중 4개가 **2022년**(10월 +0.63%p, 7월 +0.52%p …)이고,
> 가장 크게 내린 두 달은 바로 다음인 **2023년 1월(-0.39%p)·2월(-0.29%p)** 입니다. 급하게 오른 뒤 빠르게 되돌아왔습니다.

## 04-8. 연도별 평균 금리

`df["월"].dt.year`는 날짜에서 **연도만** 꺼냅니다. 연도로 묶어 평균을 내면 해마다의 수준을 비교할 수 있습니다.

In [ ]:
연평균 = df.groupby(df["월"].dt.year)["금리"].mean().round(2)
연평균

In [ ]:
fig = px.bar(x=연평균.index, y=연평균.values, text=연평균.values,
             labels={"x": "연도", "y": "평균 금리(%)"}, title="연도별 평균 예금금리 (2026년은 1~6월)")
fig.show()

## 04-9. 이동평균으로 추세 보기 ⭐ 결과물

**이동평균**은 최근 몇 개월의 평균을 한 달씩 옮겨 가며 계산한 값입니다. 들쭉날쭉한 움직임을 부드럽게 만들어 큰 흐름이 보입니다.
`rolling(3).mean()` = 최근 3개월 평균. 앞쪽 몇 달은 계산할 자료가 모자라 빈 값이 됩니다.

In [ ]:
df["3개월평균"] = df["금리"].rolling(3).mean().round(2)
df["12개월평균"] = df["금리"].rolling(12).mean().round(2)
df[["월", "금리", "3개월평균", "12개월평균"]].tail()

In [ ]:
fig = px.line(df, x="월", y=["금리", "3개월평균", "12개월평균"],
              title="예금은행 수신금리와 이동평균 (연 %)")
fig.show()

> 💡 12개월 평균선은 실제 금리보다 **늦게** 따라갑니다. 추세를 차분하게 보여 주는 대신, 방향이 바뀐 것을 늦게 알려 줍니다.
> 3개월 평균은 빨리 반응하지만 덜 부드럽습니다. 목적에 따라 기간을 고릅니다.

## 04-10. 추세 구간 표시하기

그래프에서 읽은 큰 흐름을 **구간으로 색칠**합니다. 아래 구간은 데이터의 고점·저점을 기준으로 나눈 예시이며, 강사·조별 판단으로 바꿀 수 있습니다.

In [ ]:
구간들 = [
    ("2019-01-01", "2020-08-01", "하락기", "#3776AB"),
    ("2020-08-01", "2021-05-01", "저금리 유지", "#94A3B8"),
    ("2021-05-01", "2022-11-01", "급상승기", "#E5484D"),
    ("2022-11-01", "2025-08-01", "하락기", "#3776AB"),
    ("2025-08-01", "2026-06-01", "반등", "#E5484D"),
]
fig = px.line(df, x="월", y="금리", title="예금금리 추세 구간")
for 시작, 끝, 이름, 색 in 구간들:
    fig.add_vrect(x0=시작, x1=끝, fillcolor=색, opacity=0.12, line_width=0,
                  annotation_text=이름, annotation_position="top left")
fig.show()

## 04-11. 🔧 기간과 이동평균 바꿔 보기 (실습)

`시작`, `끝`, `창`(이동평균 개월 수)만 바꾸고 다시 실행해 보세요.
예: 금리 급등기 `"2021-06-01"` ~ `"2023-06-01"`, 최근 흐름 `"2024-01-01"` ~ `"2026-06-01"`

In [ ]:
시작 = "2021-01-01"     # ← 바꿔 보세요
끝 = "2023-12-01"
창 = 6

기간 = df[(df["월"] >= 시작) & (df["월"] <= 끝)].copy()
기간[f"{창}개월평균"] = 기간["금리"].rolling(창).mean()
fig = px.line(기간, x="월", y=["금리", f"{창}개월평균"], markers=True, title=f"{시작[:7]} ~ {끝[:7]} 금리와 {창}개월 이동평균")
fig.show()

## 04-12. [선택] 1년 전 같은 달과 비교하기

`diff(12)`는 12행 앞, 즉 **1년 전 같은 달**과의 차이입니다. 계절이나 일시적 움직임을 덜 타고 1년 동안의 변화를 봅니다.

In [ ]:
df["전년동월대비"] = df["금리"].diff(12).round(2)
fig = px.bar(df, x="월", y="전년동월대비", title="1년 전 같은 달 대비 금리 변화 (%p)")
fig.show()

## 04-13. 금리 변화와 예금 유치 전략 💬 토의

아래 질문을 조별로 이야기해 봅니다. 정답이 정해진 질문이 아니라, 데이터를 보고 **업무 경험과 연결**해 보는 시간입니다.

1. 2022년처럼 금리가 빠르게 오를 때, 고객은 정기예금 **만기를 길게** 할까요, **짧게** 할까요? 우리 회사의 상품 구성은 어떻게 대응해야 할까요?
2. 2023~2025년처럼 금리가 내려갈 때, 만기가 돌아오는 고객에게 어떤 안내가 필요할까요?
3. 은행 평균 금리가 오르내릴 때, 저축은행의 예금 금리 경쟁력은 어떻게 달라질까요? 확인하려면 어떤 데이터가 더 필요할까요?

> ⚠️ 이 데이터는 은행권 **평균** 금리의 흐름입니다. 우리 회사의 수신 잔액이나 고객 이동은 포함되어 있지 않으므로, 실제 전략은 자사 수신 데이터와 함께 판단합니다.

## ✅ 오늘 배운 것

| 하고 싶은 일 | 코드 |
|---|---|
| 날짜로 읽기 | `pd.read_csv("파일", parse_dates=["date"])` |
| 선 그래프 | `px.line(df, x="월", y="금리")` |
| 최고·최저 행 찾기 | `df.loc[df["금리"].idxmax()]` |
| 전월 대비 변화 | `df["금리"].diff()` |
| 상위·하위 N개 | `df.nlargest(N, "항목")`, `df.nsmallest(N, "항목")` |
| 연도별 묶기 | `df.groupby(df["월"].dt.year)` |
| 이동평균 | `df["금리"].rolling(3).mean()` |

## 📝 복습 미션

> 아래 셀에서 **`# ←` 표시가 있는 줄만** 바꿔 실행하세요. 막히면 맨 아래 **정답 셀**을 실행해 결과를 확인합니다.

**미션 1.** 2023년의 평균 금리를 구해 보세요.
**이렇게 나오면 성공!** 3.71

**미션 2.** 6개월 이동평균을 만들고, 마지막 달(2026년 6월)의 값을 확인해 보세요.
**이렇게 나오면 성공!** 약 2.9 (2.895)

**다음 예제 전에 이것만은 꼭!**
- [ ] `diff()`는 바로 앞과의 차이, `rolling(n).mean()`은 최근 n개 평균
- [ ] 이동평균은 추세를 부드럽게 보여 주지만 **늦게** 반응한다

In [ ]:
# 미션 1
print(round(df[df["월"].dt.year == 2022]["금리"].mean(), 2))   # ← 2022를 2023으로

In [ ]:
# 미션 2
창 = 3        # ← 3을 6으로
df["이동평균"] = df["금리"].rolling(창).mean()
df[["월", "금리", "이동평균"]].tail(1)

### 정답 (먼저 위 셀을 바꿔 실행해 본 뒤 확인하세요)

In [ ]:
# 미션 1 정답 (방법 1: 04-8의 연평균 표에서 찾기)
print(연평균[2023])

In [ ]:
# 미션 1 정답 (방법 2: 2023년 행만 골라 평균)
print(round(df[df["월"].dt.year == 2023]["금리"].mean(), 2))

In [ ]:
# 미션 2 정답
df["6개월평균"] = df["금리"].rolling(6).mean()
df[["월", "금리", "6개월평균"]].tail(1)

---
### ▶ 다음 예제 05. LSTM 금융 시계열 예측 체험
오늘 본 금리 데이터를 그대로 이어 씁니다. AI 모델(LSTM)이 "다음 달 금리"를 어떻게 예측하는지, 그리고 **왜 틀리는지** 직접 확인합니다.